# Tourism Package Purchase Prediction

**Project 2 · Advanced Machine Learning and MLOps**

## Business objective

“Visit with Us” wants to identify customers who may be interested in its Wellness Tourism Package and prioritize outreach. This project builds a reproducible prediction pipeline, evaluates it on held-out data, and packages the model in a simple app.

## Rubric coverage

| Rubric area | Marks | Project evidence |
|---|---:|---|
| Data registration and validation | 3 | Repository CSV, schema checks, printed summary |
| Data preparation | 7 | Cleaning, leakage review, stratified splits, workflow artifact |
| Model building and experimentation | 13 | Tuned Random Forest, cross-validation, saved parameters and metrics |
| Model deployment | 11 | Streamlit app and dependencies |
| GitHub Actions MLOps pipeline | 16 | Automated validation → preparation → training → model commit |
| Output evaluation evidence | 4 | GitHub structure/workflow and public app screenshots and links |
| Notebook quality | 6 | Organized analysis, executed outputs, findings, and limitations |

> **Reproducibility:** The analysis uses the shared utilities in `src/` and the registered source file in `data/`.


## 1. Setup and dataset registration

The source file is kept in the repository’s `data/` folder. The data-validation function checks the expected schema and binary target, then reports missing values, duplicates, and class balance.


In [ ]:
# Locate the project root when the folder is extracted in Colab or run locally.
from pathlib import Path
import sys

possible_roots = [Path.cwd(), *Path.cwd().parents, Path("/content/Project_2_Tourism_MLOps")]
PROJECT_ROOT = next((p for p in possible_roots if (p / "src" / "data_validation.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Extract the complete Project_2_Tourism_MLOps folder, then open this notebook from its notebooks folder.")
sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / "data" / "tourism.csv"


In [ ]:
# Reuse the exact validation code called by the GitHub Actions workflow.
from src.data_validation import validate_dataset

df = validate_dataset(DATA_PATH)


**Dataset review.** The supplied file includes a saved index column, which registration removes. It contains 4,128 customer records and 20 meaningful fields, with no missing values or exact duplicate rows. The target is imbalanced: 797 customers purchased a package and 3,331 did not. Because the positive class is smaller, evaluation will include recall and average precision as well as ROC-AUC; accuracy alone would be misleading.

## 2. Exploratory analysis

First check the target balance and compare purchase rates across a few customer-profile fields. These summaries help explain the data and do not determine the model features by themselves.


In [ ]:
# Show the target balance and a few profile-based purchase rates.
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df["ProdTaken"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color=["#78909C", "#2E7D32"])
axes[0].set_title("Package purchase counts")
axes[0].set_xlabel("ProdTaken (0 = no, 1 = yes)")
axes[0].set_ylabel("Customers")
for ax, feature in zip(axes[1:], ["Passport", "CityTier"]):
    (df.groupby(feature)["ProdTaken"].mean() * 100).plot(kind="bar", ax=ax, color="#2F6B8A")
    ax.set_title(f"Purchase rate by {feature}")
    ax.set_ylabel("Customers purchasing (%)")
    ax.set_xlabel(feature)
plt.tight_layout()
plt.show()

print("Numeric summary:")
display(df.select_dtypes(include="number").describe().T)


**EDA observation.** Purchases are the minority outcome: 797 of 4,128 customers (19.3%) purchased a package. Purchase rates differ across the profile groups shown: 35.8% for customers with a passport versus 12.4% for those without one; city-tier rates range from 16.5% in Tier 1 to 26.5% in Tier 2, with Tier 3 at 24.1%. The Tier 2 group is much smaller than the others, so its rate should be interpreted cautiously. These are descriptive patterns in historical data, not causal effects or guarantees about individual customers.

## 3. Cleaning, feature selection, and data preparation

The business question is to estimate purchase interest before a sales contact. The file includes interaction fields recorded during or after a pitch, so the model excludes `TypeofContact`, `DurationOfPitch`, `NumberOfFollowups`, `ProductPitched`, and `PitchSatisfactionScore`. It also drops `CustomerID`, which identifies a record but should not predict buying behavior. This avoids giving the model information that would not be available at the intended decision point.

The pipeline standardizes the obvious label variations `Fe Male` and `Free Lancer`, keeps other recorded categories intact, and does not remove unusual trip counts without a verified data-quality rule. The 80/20 split is stratified so both sets retain a similar purchase rate.


In [ ]:
# Reuse the same cleaning and split function as the automated workflow.
from src.prepare_data import make_splits, clean_data, POST_CONTACT_COLUMNS, IDENTIFIER_COLUMNS

train_df, test_df = make_splits(df, output_dir=PROJECT_ROOT / "artifacts" / "processed", test_size=0.20, seed=42)


In [ ]:
# Confirm that the split sizes and positive-class proportions remain close.
split_summary = pd.DataFrame({
    "rows": [len(train_df), len(test_df)],
    "purchase_rate": [train_df["ProdTaken"].mean(), test_df["ProdTaken"].mean()],
}, index=["train", "test"])
display(split_summary)
print("Model inputs:", [c for c in train_df.columns if c != "ProdTaken"])


## 4. Model building and experiment tracking

The model uses a preprocessing pipeline that imputes missing numeric and categorical values and one-hot encodes categories. A class-weighted Random Forest is tuned with 3-fold stratified cross-validation. The best setting is selected by ROC-AUC on training folds; the test set is held back for a single final evaluation.

The script records every grid-search result, the best parameters, holdout metrics, the confusion matrix, and a classification report. It saves the fitted preprocessing-and-model pipeline as one file so the deployed app applies the same transformations used in training.


In [ ]:
# Train and evaluate using the repository module also used by GitHub Actions.
from src.train_model import train_and_evaluate

best_model, metrics, search = train_and_evaluate(
    train_path=PROJECT_ROOT / "artifacts" / "processed" / "train.csv",
    test_path=PROJECT_ROOT / "artifacts" / "processed" / "test.csv",
    model_path=PROJECT_ROOT / "models" / "best_model.joblib",
    report_dir=PROJECT_ROOT / "reports",
)


In [ ]:
# Review the tuned configurations and saved holdout summary.
cv_results = pd.read_csv(PROJECT_ROOT / "reports" / "cv_results.csv")
metrics_table = pd.Series(metrics, name="holdout value").to_frame()
display(cv_results.sort_values("rank_test_score").head(5))
display(metrics_table)


In [ ]:
# Summarize the holdout results in plain language from the measured metrics.
print(
    f"On the held-out test set, ROC-AUC was {metrics['roc_auc']:.3f}, "
    f"average precision was {metrics['average_precision']:.3f}, "
    f"recall was {metrics['recall']:.3f}, and precision was {metrics['precision']:.3f}."
)
print(
    "The model can help rank customers for human-reviewed outreach. "
    "The classification threshold should be chosen with campaign capacity and the cost of missed or unnecessary contacts in mind."
)


## 5. Deployment and MLOps workflow

The Streamlit app loads `models/best_model.joblib`, collects the model’s pre-contact customer fields into a dataframe, and displays a purchase-likelihood estimate. The GitHub Actions workflow runs when code is pushed to `main` or started manually. It validates the registered CSV, prepares train/test splits, passes those splits to the training job as a workflow artifact, tunes and evaluates the model, then commits the model and experiment records to `main`.

The project page’s rubric names Streamlit Community Cloud; its submission guidance also asks for a public Hugging Face Space. The deployment guide supports both targets. The Hugging Face package uses Docker because the current Spaces documentation marks the built-in Streamlit SDK as deprecated.

## 6. Conclusion and limitations

Use the reported holdout metrics to decide whether the model is useful for prioritizing outreach, with particular attention to recall, precision, and average precision because purchases are the minority class. A probability estimate is not a guarantee of behavior, and a 0.5 threshold is only a starting point.

This is a single historical dataset. The model may not reflect current customer behavior, and its predictions can be affected by class imbalance, data quality, and unmeasured factors. Excluding post-contact fields makes the task realistic, but it may also limit predictive performance. Before operational use, validate performance on newer data, review results across customer groups, and keep a person involved in outreach decisions.

## Deployment evidence

The project submission includes the public GitHub repository, the Streamlit Community Cloud app, and a Hugging Face Space, with screenshots of the repository structure, successful workflow run, and deployed application. These links and screenshots are included with the final executed HTML report.
